# Sistema Híbrido de Recomendaciones: ALS + K-Means

**Mismo pipeline que `modelo_recomendaciones.ipynb` pero reemplazando SVD por ALS.**

---

**Por qué SÍ usar ALS acá:**
- Nuestros datos son feedback implícito (evento → peso 1/2/3). SVD trata el cero como "rating = 0", ALS lo trata como "no visto" — mucho más correcto.
- La biblioteca `implicit` está optimizada para matrices sparse grandes.

**Por qué NO (o tener cuidado):**
- Necesita la dependencia `implicit` (`pip install implicit`).
- Más hiperparámetros para tunear.
- El modelo no da varianza explicada directamente → hay que evaluar con otras métricas.
- Si la matriz es extremadamente sparse (99.99%+), la diferencia práctica puede ser marginal.

---

**Objetivo:** Recomendar productos según estado del usuario:
- **Cold-start (0 eventos)**: Cluster-based recommendations
- **Warm-start light (1-5 eventos)**: Blend cluster + ALS
- **Warm-start (>5 eventos)**: Pure ALS collaborative filtering

**Inputs:**
- `data/final/events_final.csv`
- `data/final/user_cluster_map.csv`
- `data/final/recomendaciones_cold_start.csv`
- `data/final/user_clustering_gmm_results.csv`

In [4]:
import os
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pickle
import json
from scipy.sparse import csr_matrix

try:
    from implicit.als import AlternatingLeastSquares
    print("implicit instalado correctamente")
except ImportError:
    raise ImportError(
        "Instalá implicit primero:\n"
        "  pip install implicit\n"
        "  # o si usás conda: conda install -c conda-forge implicit"
    )

# Configuraciones generales
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
RANDOM_STATE = 42

print("=== IMPORTACIONES HECHAS ===")

implicit instalado correctamente
=== IMPORTACIONES HECHAS ===


---
## Sección 1: Cargar Clustering & Datos de Eventos

In [6]:
from pathlib import Path

# Locate project root regardless of where the notebook server is launched from
_here = Path(os.path.abspath(""))
_project_root = next(
    p for p in [_here, _here.parent, _here.parent.parent]
    if (p / "data" / "final").exists()
)

events_path = _project_root / "data" / "final" / "events_final.csv"

df_events = pd.read_csv(events_path)
df_events['event_time'] = pd.to_datetime(df_events['event_time'], utc=True)

# ── Pesos por tipo de evento ─────────────────────────────────────────────────
weight_map = {'view': 1, 'cart': 2, 'purchase': 3}
df_events['peso'] = df_events['event_type'].map(weight_map).fillna(0)

# ── Interacciones con peso simple ────────────────────────────────────────────
df_interactions = (
    df_events
    .groupby(['user_id', 'product_id'], as_index=False)['peso']
    .sum()
)
df_interactions = df_interactions[df_interactions['peso'] > 0].reset_index(drop=True)

# ── Interacciones con score temporal ─────────────────────────────────────────
# score_temporal = peso * exp(-0.01 * dias_desde_ultima_interaccion)
fecha_ref = df_events['event_time'].max()
ultima_interaccion = (
    df_events.groupby(['user_id', 'product_id'])['event_time']
    .max()
    .reset_index()
    .rename(columns={'event_time': 'ultima_interaccion'})
)
df_interactions_temporal = df_interactions.merge(ultima_interaccion, on=['user_id', 'product_id'])
df_interactions_temporal['dias'] = (fecha_ref - df_interactions_temporal['ultima_interaccion']).dt.days
df_interactions_temporal['score_temporal'] = (
    df_interactions_temporal['peso'] * np.exp(-0.01 * df_interactions_temporal['dias'])
).round(4)
df_interactions_temporal = df_interactions_temporal[df_interactions_temporal['score_temporal'] > 0].reset_index(drop=True)

print(f"Largo de los eventos:                    {df_events.shape}")
print(f"Largo de las interacciones (peso):       {df_interactions.shape}")
print(f"Largo de las interacciones (temporal):   {df_interactions_temporal.shape}")

# ── Contar eventos por usuario para determinar tier ─────────────────────────
user_event_count = df_events.groupby('user_id').size().reset_index(name='event_count')

def assign_tier(n):
    if n <= 5:
        return 1
    return 2

user_event_count['tier'] = user_event_count['event_count'].apply(assign_tier)

print("\nDISTRIBUCION DE USUARIOS POR TIER")
print(f"Tier 1 (1-5 eventos):  {(user_event_count['tier']==1).sum():,}")
print(f"Tier 2 (>5 eventos):   {(user_event_count['tier']==2).sum():,}")


Largo de los eventos:                    (884312, 17)
Largo de las interacciones (peso):       (556637, 3)
Largo de las interacciones (temporal):   (556637, 6)

DISTRIBUCION DE USUARIOS POR TIER
Tier 1 (1-5 eventos):  382,745
Tier 2 (>5 eventos):   24,492


---
## Sección 2: Construir Modelo ALS en Matriz User-Item

**Diferencia clave con SVD:**
La biblioteca `implicit` espera una matriz **item-usuario** (transpuesta). Internamente aplica confidence weighting: `C_ui = 1 + alpha * r_ui`, donde `r_ui` es el score.

ALS minimiza:
$$\sum_{u,i} c_{ui}(r_{ui} - x_u^T y_i)^2 + \lambda(\|x_u\|^2 + \|y_i\|^2)$$

Donde `c_ui = 1` si no hubo interacción y `c_ui = 1 + alpha * score` si la hubo.

In [ ]:
N_FACTORS = 50
N_ITERATIONS = 50
REGULARIZATION = 0.01
ALPHA = 40  # confidence scaling: C = 1 + alpha * score

# Crear mapeos de IDs (iguales para ambos modelos)
unique_users    = sorted(df_interactions['user_id'].unique())
unique_products = sorted(df_interactions['product_id'].unique())
user_id_map     = {uid: idx for idx, uid in enumerate(unique_users)}
product_id_map  = {pid: idx for idx, pid in enumerate(unique_products)}

N_USERS = len(unique_users)
N_ITEMS = len(unique_products)

def construir_matriz_user_item(df, columna_score):
    """Construye una matriz sparse usuario-producto."""
    row  = df['user_id'].map(user_id_map).values
    col  = df['product_id'].map(product_id_map).values
    data = df[columna_score].values.astype(np.float32)
    return csr_matrix((data, (row, col)), shape=(N_USERS, N_ITEMS))

# ── Modelo A: peso simple ─────────────────────────────────────────────────────
matriz_peso_ui = construir_matriz_user_item(df_interactions, 'peso')
matriz_peso_iu = matriz_peso_ui.T.tocsr()  # implicit espera item-usuario

als_peso = AlternatingLeastSquares(
    factors=N_FACTORS,
    regularization=REGULARIZATION,
    iterations=N_ITERATIONS,
    random_state=RANDOM_STATE,
    use_gpu=False,
)
print("Entrenando ALS con peso simple...")
als_peso.fit(matriz_peso_iu * ALPHA)

# ── Modelo B: score temporal ──────────────────────────────────────────────────
df_temporal_alineado = df_interactions_temporal[
    df_interactions_temporal['user_id'].isin(unique_users) &
    df_interactions_temporal['product_id'].isin(unique_products)
].copy()
matriz_temporal_ui = construir_matriz_user_item(df_temporal_alineado, 'score_temporal')
matriz_temporal_iu = matriz_temporal_ui.T.tocsr()

als_temporal = AlternatingLeastSquares(
    factors=N_FACTORS,
    regularization=REGULARIZATION,
    iterations=N_ITERATIONS,
    random_state=RANDOM_STATE,
    use_gpu=False,
)
print("Entrenando ALS con score temporal...")
als_temporal.fit(matriz_temporal_iu * ALPHA)

print(f"\nMatriz user-item: {matriz_peso_ui.shape}")
print(f"Sparsidad: {1 - matriz_peso_ui.nnz / (matriz_peso_ui.shape[0] * matriz_peso_ui.shape[1]):.4f}")
print(f"\nShapes del modelo ALS (peso simple):")
print(f"  user_factors: {als_peso.user_factors.shape}")
print(f"  item_factors: {als_peso.item_factors.shape}")

# implicit nombra los factores segun las dimensiones de la matriz de entrada (item-usuario).
# Segun la version instalada, user_factors puede corresponder a items o a usuarios.
# Detectamos cual es cual en runtime para no depender de la convencion de la version.
def _user_vec(model, user_idx):
    """Vector de factores de un usuario dado su indice."""
    if model.user_factors.shape[0] == N_USERS:
        return model.user_factors[user_idx]
    return model.item_factors[user_idx]

def _item_matrix(model):
    """Matriz de factores de todos los items (N_ITEMS x n_factors)."""
    if model.user_factors.shape[0] == N_ITEMS:
        return model.user_factors
    return model.item_factors

print(f"\nConvencion detectada:")
print(f"  _user_vec usa {'user_factors' if als_peso.user_factors.shape[0]==N_USERS else 'item_factors'}")
print(f"  _item_matrix usa {'user_factors' if als_peso.user_factors.shape[0]==N_ITEMS else 'item_factors'}")

Entrenando ALS con peso simple...


 90%|█████████ | 45/50 [00:24<00:02,  1.90it/s]

In [ ]:
# ── Comparar modelos por RMSE de reconstrucción en muestra aleatoria ─────────

def rmse_muestra(als_model, matriz_ui, n_sample=10_000):
    """RMSE de reconstruccion sobre entradas no-cero de la matriz."""
    coo   = matriz_ui.tocoo()
    idx   = np.random.default_rng(RANDOM_STATE).choice(
        len(coo.data), size=min(n_sample, len(coo.data)), replace=False
    )
    u_idx = coo.row[idx]   # indices de usuario
    i_idx = coo.col[idx]   # indices de item
    real  = coo.data[idx]
    # Usar los helpers que detectan la convencion de la version de implicit
    item_mat = _item_matrix(als_model)   # (N_ITEMS, n_factors)
    pred = np.array([
        _user_vec(als_model, u) @ item_mat[i]
        for u, i in zip(u_idx, i_idx)
    ])
    return float(np.sqrt(np.mean((real - pred) ** 2)))

rmse_peso     = rmse_muestra(als_peso,     matriz_peso_ui)
rmse_temporal = rmse_muestra(als_temporal, matriz_temporal_ui)

print(f"{'Modelo':<30} {'RMSE (muestra 10k)':>22}")
print(f"{'ALS con peso simple':<30} {rmse_peso:>22.4f}")
print(f"{'ALS con score temporal':<30} {rmse_temporal:>22.4f}")
print(f"\nDiferencia: {(rmse_temporal - rmse_peso):+.4f}  (menor es mejor)")

if rmse_temporal <= rmse_peso:
    als_model = als_temporal
    matriz_ui = matriz_temporal_ui
    print("\n Se usa ALS con score temporal")
else:
    als_model = als_peso
    matriz_ui = matriz_peso_ui
    print("\n Se usa ALS con peso simple")

Modelo                             RMSE (muestra 10k)
ALS con peso simple                            3.0764
ALS con score temporal                         1.9535

Diferencia: -1.1229  (menor es mejor)

 Se usa ALS con score temporal


---
## Sección 3: Definir Tiers y Crear Diccionarios de Mapeo

In [ ]:
user_cluster_map_path = _project_root / 'data' / 'final' / 'user_cluster_map.csv'
if not user_cluster_map_path.exists():
    raise FileNotFoundError(
        f'No existe {user_cluster_map_path}. '
        'Ejecuta Clustering_Kmeans.ipynb primero.'
    )

df_user_cluster = pd.read_csv(user_cluster_map_path)
user_cluster_map = dict(zip(df_user_cluster['user_id'], df_user_cluster['cluster_id']))
print(f'Usuarios con cluster asignado: {len(user_cluster_map):,}')

# user_profile: user_id -> (tier, cluster_id, user_idx)
# Guardamos user_idx en lugar de latent_factors porque ALS los tiene en als_model.user_factors
user_profile = {}

for user_id in unique_users:
    tier_row = user_event_count[user_event_count['user_id'] == user_id]['tier'].values
    tier = int(tier_row[0]) if len(tier_row) > 0 else 0
    cluster = user_cluster_map.get(user_id, -1)
    user_idx = user_id_map[user_id]

    user_profile[user_id] = {
        'tier': tier,
        'cluster_id': cluster,
        'user_idx': user_idx,
    }

print(f'User profiles creados: {len(user_profile):,}')

cold_start_path = _project_root / 'data' / 'final' / 'recomendaciones_cold_start.csv'
if cold_start_path.exists():
    cold_start_recs = pd.read_csv(cold_start_path)
    print(f'Recomendaciones cold-start cargadas: {cold_start_recs.shape[0]} filas')
else:
    raise FileNotFoundError(
        f'No existe {cold_start_path}. '
        'Ejecuta Clustering_Kmeans.ipynb primero.'
    )

Usuarios con cluster asignado: 407,237
User profiles creados: 407,237
Recomendaciones cold-start cargadas: 40 filas


In [ ]:
gmm_results_path = _project_root / 'data' / 'final' / 'user_clustering_gmm_results.csv'
if gmm_results_path.exists():
    df_gmm_map = pd.read_csv(gmm_results_path, usecols=['user_id', 'gmm_cluster'])
    user_gmm_map = dict(zip(df_gmm_map['user_id'], df_gmm_map['gmm_cluster']))
    n_activos = sum(v == 0 for v in user_gmm_map.values())
    n_pasivos = sum(v == 1 for v in user_gmm_map.values())
    print(f'GMM clusters cargados: {len(user_gmm_map):,} usuarios')
    print(f'  Activos  (0): {n_activos:,} ({n_activos / len(user_gmm_map) * 100:.1f}%)')
    print(f'  Pasivos  (1): {n_pasivos:,} ({n_pasivos / len(user_gmm_map) * 100:.1f}%)')
else:
    user_gmm_map = {}
    print('!!!!!!!!!! No existe user_clustering_gmm_results.csv.')
    print('   Tier 1 usara pesos fijos 0.7/0.3 (modo pasivo por defecto).')

GMM clusters cargados: 407,237 usuarios
  Activos  (0): 56,351 (13.8%)
  Pasivos  (1): 350,886 (86.2%)


---
## Sección 4: Función de Recomendación Híbrida

**Diferencia con SVD:** En lugar de `svd_model.components_.T @ latent_factors`, el score ALS se calcula como:
```
scores = als_model.item_factors @ als_model.user_factors[user_idx]
```
Ambos son productos punto — la lógica de negocio queda idéntica.

In [ ]:
def recomendar_nuevo_usuario(n=10):
    """Fallback para usuarios que no existen en ningun dataset."""
    cluster_default = int(cold_start_recs['cluster_id'].value_counts().idxmax())
    recs = cold_start_recs[cold_start_recs['cluster_id'] == cluster_default].head(n).copy()
    recs = recs[['product_id', 'score_cluster']].rename(columns={'score_cluster': 'score'})
    recs['method'] = 'global-popular'
    return recs.reset_index(drop=True)


def recomendar(user_id, n=10):
    """
    Recomendacion hibrida por tier.

    Tier 0: sin historial  -> recomendaciones por cluster (cold start)
    Tier 1: 1-5 eventos    -> hibrido cluster + ALS, pesos ajustados por GMM
    Tier 2: >5 eventos     -> ALS puro
    """
    if user_id not in user_profile:
        return recomendar_nuevo_usuario(n=n)

    profile    = user_profile[user_id]
    tier       = profile['tier']
    cluster_id = profile['cluster_id']
    user_idx   = profile['user_idx']

    # ── TIER 0: sin historial ──────────────────────────────────────────────
    if tier == 0:
        if cluster_id < 0:
            return recomendar_nuevo_usuario(n=n)
        recs = cold_start_recs[cold_start_recs['cluster_id'] == cluster_id].head(n).copy()
        recs = recs[['product_id', 'score_cluster']].rename(columns={'score_cluster': 'score'})
        recs['method'] = 'cluster-based'
        return recs.reset_index(drop=True)

    # ── TIER 1: hibrido con pesos ajustados por GMM ────────────────────────
    elif tier == 1:
        gmm_cluster = user_gmm_map.get(user_id, 1)
        alfa, beta  = (0.4, 0.6) if gmm_cluster == 0 else (0.7, 0.3)
        tipo        = 'activo' if gmm_cluster == 0 else 'pasivo'

        cluster_recs = cold_start_recs[cold_start_recs['cluster_id'] == cluster_id].copy()

        user_vec   = _user_vec(als_model, user_idx)       # (n_factors,)
        item_mat   = _item_matrix(als_model)               # (N_ITEMS, n_factors)
        als_scores = item_mat @ user_vec                   # (N_ITEMS,)
        als_df = pd.DataFrame({'product_id': unique_products, 'als_score': als_scores})

        blended = cluster_recs[['product_id', 'score_cluster']].merge(als_df, on='product_id', how='outer')
        blended['score_cluster'] = blended['score_cluster'].fillna(0)
        blended['als_score']     = blended['als_score'].fillna(0)

        c_min, c_max = blended['score_cluster'].min(), blended['score_cluster'].max()
        s_min, s_max = blended['als_score'].min(),     blended['als_score'].max()
        blended['score_cluster_norm'] = (blended['score_cluster'] - c_min) / (c_max - c_min + 1e-6)
        blended['als_score_norm']     = (blended['als_score']     - s_min) / (s_max - s_min + 1e-6)

        blended['score'] = alfa * blended['score_cluster_norm'] + beta * blended['als_score_norm']
        blended = blended.sort_values('score', ascending=False).head(n)
        blended['method'] = f'hybrid(a={alfa},b={beta},{tipo})'
        return blended[['product_id', 'score', 'method']].reset_index(drop=True)

    # ── TIER 2: ALS puro ───────────────────────────────────────────────────
    else:
        user_vec   = _user_vec(als_model, user_idx)
        item_mat   = _item_matrix(als_model)
        als_scores = item_mat @ user_vec
        recs = pd.DataFrame({'product_id': unique_products, 'score': als_scores})
        recs = recs.sort_values('score', ascending=False).head(n)
        recs['method'] = 'als'
        return recs[['product_id', 'score', 'method']].reset_index(drop=True)


print("Funciones de recomendacion listas!")

Funciones de recomendacion listas!


---
## Sección 5: Testear Pipeline de Recomendaciones

In [ ]:
print("TESTING RECOMENDACIONES POR TIER \n")

# Tier 0
tier0_users = user_event_count[user_event_count['tier'] == 0]['user_id'].head(1).values
if len(tier0_users) > 0:
    print(f"TIER 0 (Cold-start): user_id={tier0_users[0]}")
    print(recomendar(tier0_users[0], n=10))
    print()

# Tier 1
tier1_users = user_event_count[user_event_count['tier'] == 1]['user_id'].head(1).values
if len(tier1_users) > 0:
    print(f"TIER 1 (Warm-light): user_id={tier1_users[0]}")
    print(recomendar(tier1_users[0], n=10))
    print()

# Tier 2
tier2_users = user_event_count[user_event_count['tier'] == 2]['user_id'].head(1).values
if len(tier2_users) > 0:
    print(f"TIER 2 (Warm): user_id={tier2_users[0]}")
    print(recomendar(tier2_users[0], n=10))
    print()

# Usuario completamente nuevo
print("USUARIO NUEVO (no en sistema):")
print(recomendar_nuevo_usuario(n=10))

TESTING RECOMENDACIONES POR TIER 

TIER 1 (Warm-light): user_id=1515915625353226922
   product_id     score                      method
0     1785245  0.767763  hybrid(a=0.7,b=0.3,pasivo)
1     3642540  0.550112  hybrid(a=0.7,b=0.3,pasivo)
2      246841  0.418126  hybrid(a=0.7,b=0.3,pasivo)
3      194199  0.327426  hybrid(a=0.7,b=0.3,pasivo)
4     1785246  0.316743  hybrid(a=0.7,b=0.3,pasivo)
5     1724958  0.299993  hybrid(a=0.7,b=0.3,pasivo)
6      802811  0.296730  hybrid(a=0.7,b=0.3,pasivo)
7     4101482  0.287736  hybrid(a=0.7,b=0.3,pasivo)
8      243090  0.281180  hybrid(a=0.7,b=0.3,pasivo)
9     1850374  0.279447  hybrid(a=0.7,b=0.3,pasivo)

TIER 2 (Warm): user_id=1515915625353230683
   product_id     score method
0     1724625  0.383189    als
1     1724626  0.354976    als
2     1271549  0.354328    als
3      471287  0.352356    als
4      124712  0.317300    als
5     1271550  0.306536    als
6      207056  0.297075    als
7      460841  0.276116    als
8     1777246  0.2631

---
## Sección 6: Exportar Modelo y Mapeos para Producción

In [ ]:
_models_dir = _project_root / 'models'
_models_dir.mkdir(exist_ok=True)

# 1. Guardar modelo ALS
als_model_path = _models_dir / 'als_model.pkl'
with open(als_model_path, 'wb') as f:
    pickle.dump(als_model, f)
print(f" ALS model guardado: {als_model_path}")

# 2. Guardar user_profile
user_profile_path = _models_dir / 'user_profile_als.pkl'
with open(user_profile_path, 'wb') as f:
    pickle.dump(user_profile, f)
print(f" User profiles guardados: {user_profile_path}")

# 3. Guardar mapeos de IDs
id_maps = {
    'user_id_map': user_id_map,
    'product_id_map': product_id_map,
    'reverse_product_id_map': {v: k for k, v in product_id_map.items()}
}
id_maps_path = _models_dir / 'id_maps_als.pkl'
with open(id_maps_path, 'wb') as f:
    pickle.dump(id_maps, f)
print(f" ID maps guardados: {id_maps_path}")

# 4. Guardar model info
tier_dist = user_event_count.groupby('tier').size().to_dict()
tier_info = {
    'model_type': 'ALS',
    'tier_distribution': tier_dist,
    'total_users': len(user_event_count),
    'n_factors': N_FACTORS,
    'n_iterations': N_ITERATIONS,
    'regularization': REGULARIZATION,
    'alpha': ALPHA,
}
with open(_models_dir / 'als_model_info.json', 'w') as f:
    json.dump(tier_info, f, indent=2)
print(f" Model info guardada: {_models_dir / 'als_model_info.json'}")

# 5. Exportar user-tier mapping
user_tier_export = user_event_count[['user_id', 'event_count', 'tier']].copy()
user_tier_export['cluster_id'] = user_tier_export['user_id'].map(user_cluster_map)
user_tier_export.to_csv(_project_root / 'data' / 'final' / 'user_tier_mapping_als.csv', index=False)
print(f" User-tier mapping guardada: {_project_root / 'data' / 'final' / 'user_tier_mapping_als.csv'}")

 ALS model guardado: ../models/als_model.pkl
 User profiles guardados: ../models/user_profile_als.pkl
 ID maps guardados: ../models/id_maps_als.pkl
 Model info guardada: ../models/als_model_info.json
 User-tier mapping guardada: ../data/final/user_tier_mapping_als.csv
